# Dipole Angle vs Parallactic Angle Correlation

This notebook demonstrates that the **raw dipole angle** (`r:dipoleAngle`) measured by the
Rubin/LSST AP pipeline is fully correlated with the **parallactic angle** η computed from
first principles (RA, Dec, MJD → hour angle → arctan2 formula).

Both quantities peak at **−90° and +90°** relative to the North–South direction, which
confirms that the dipoles are driven by atmospheric differential refraction / PSF elongation
along the zenith direction.

## Angle conventions used

| Column | Convention | Range |
|--------|------------|-------|
| `r:dipoleAngle` | CCW from East (+x pixel axis) | −180 … +180° or 0 … 360° |
| `parallactic_angle_deg` | CCW from North (astronomical PA) | −180 … +180° |
| `azimuth_deg` | CW from North (astropy standard) | 0 … 360° |
| `zenith_angle_deg` | 90° − altitude | 0 … 90° |
| `sin_zenith` | sin(zenith_angle_deg) | 0 … 1 |
| `airmass` | ≈ 1/cos(z) | ≥ 1 |
| `hour_angle_hr` | H = LST − RA, wrapped to (−12h, +12h] | ±12 h |
| `hour_angle_deg` | same, in degrees | ±180° |

**No dipole_PA conversion is needed** — we work directly with `r:dipoleAngle`
and the parallactic angle in their native ranges.

## Physical picture

Atmospheric differential chromatic refraction (DCR) displaces each source along the
great circle toward the zenith.  In the tangent plane this direction is exactly the
parallactic angle η.  Because the AP pipeline subtracts a template taken at a
**different** airmass (and possibly a different hour angle), the residual PSF elongation
generates a dipole whose axis tracks η.  The **amplitude** of DCR scales as
tan z ≈ sin z for moderate zenith angles, so `r:dipoleLength` should grow with sin z.

The algorithm may orient the positive lobe of the dipole in either direction along η
(the sign is not physically meaningful), hence:
* the signed difference `delta_dipole_para = r:dipoleAngle − η` clusters near both 0° and ±180°;
* the headless (folded) difference `delta_dipole_para_folded` is the minimum of
  |Δ| and 180° − |Δ| and clusters near 0°.

## Strategy

* Load dipole alerts from `data_DIPOLES_01c/` parquet files.
* Compute η, H, azimuth, zenith, sin z and airmass for every alert with `astropy`.
* 360° rose diagrams stacked by band, 2×3 DDF grid: `r:dipoleAngle`, azimuth, η, Δ signed, Δ folded.
* Bar-plot distributions of zenith and airmass (stacked by band, 2×3 DDF).
* `r:dipoleLength` vs sin z — scatter + median profile ± MAD (2×3 DDF + all-DDFs panel).
* `r:dipoleLength` vs |H| and `r:dipoleAngle` vs H — scatter + median profile (2×3 DDF + combined).
* Pearson/Spearman correlation table and Spearman ρ heatmaps.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-30 : add sin(z) and hour-angle analyses, Δ signed + folded rose diagrams
- last update : 2026-05-31 : show theory
- last update: 2026-06-02 : add the dipole length theory
- last update: 2026-06-02 : aplot scatter vs theory per band related to dipole length

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u


from scipy.integrate import simpson

from speclite import filters  # retreive LSST bands
import ref_index  # retrieve refractive index


import numpy.typing as npt
from typing import Callable


warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"
NB_TAG = "DIPOLES_05b"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST – Cerro Pachón ─────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Observatory: lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}
DDF_NAMES = list(DEEP_FIELDS.keys())  # fixed order for 2×3 grid

# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# -- Wavelength table to compute dispersion on refractive index ---
# Wavelength grid covering the full LSST range (0.3 – 1.1 µm)
LAM = np.linspace(0.3, 1.1, 3000)  # µm

# ── Conversion constants ───────────────────────────────────────────────────────
RAD_TO_ARCSEC = 180.0 / np.pi * 3600.0


# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
def profile_median(x, y, bins, min_count=1, return_counts=False):
    """
    Compute a median profile of y as a function of x.

    Parameters
    ----------
    x : array-like
        X values
    y : array-like
        Y values
    bins : array-like
        Bin edges in X
    min_count : int, optional
        Minimum number of points per bin to compute stats
    return_counts : bool, optional
        If True, also return counts per bin

    Returns
    -------
    x_centers : ndarray
        Bin centers
    y_median : ndarray
        Median of Y in each bin
    y_err : ndarray
        Error on median = RMS / sqrt(N)
    y_rms : ndarray
        RMS around the median
    (optional) counts : ndarray
        Number of points per bin
    """

    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    if x.shape[0] != y.shape[0]:
        raise ValueError(f"x and y mismatch: {x.shape} vs {y.shape}")

    bins = np.asarray(bins)

    # Digitize
    bin_indices = np.digitize(x, bins) - 1
    nbins = len(bins) - 1

    # Outputs
    y_median = np.full(nbins, np.nan)
    y_rms = np.full(nbins, np.nan)
    y_err = np.full(nbins, np.nan)
    counts = np.zeros(nbins, dtype=int)

    for i in range(nbins):
        mask = bin_indices == i
        yi = y[mask]

        if yi.size >= min_count:
            med = np.median(yi)
            mad = np.median(np.abs(yi - med))
            # rms = np.sqrt(np.mean((yi - med) ** 2))
            rms = 1.4826 * mad
            n = yi.size

            y_median[i] = med
            y_rms[i] = rms
            y_err[i] = rms / np.sqrt(n)
            counts[i] = n

    # Bin centers
    x_centers = 0.5 * (bins[:-1] + bins[1:])

    if return_counts:
        return x_centers, y_median, y_err, y_rms, counts
    else:
        return x_centers, y_median, y_err, y_rms

In [ ]:
def profile_quantile(x, y, bins, quantiles=(0.16, 0.5, 0.84), min_count=5, return_counts=False):
    """
    Profile Y vs X using quantiles (seaborn / corner style).

    Parameters
    ----------
    x, y : array-like
    bins : array-like
        Bin edges
    quantiles : tuple
        Quantiles to compute (default: 16%, 50%, 84%)
    min_count : int
        Minimum points per bin
    return_counts : bool

    Returns
    -------
    x_centers
    q_values : dict
        Keys are quantiles (e.g. 0.16, 0.5, 0.84)
    counts (optional)
    """

    x = np.asarray(x)
    y = np.asarray(y)

    bin_indices = np.digitize(x, bins) - 1
    nbins = len(bins) - 1

    q_values = {q: np.full(nbins, np.nan) for q in quantiles}
    counts = np.zeros(nbins, dtype=int)

    for i in range(nbins):
        mask = bin_indices == i
        yi = y[mask]

        if yi.size >= min_count:
            qs = np.quantile(yi, quantiles)
            for q, val in zip(quantiles, qs):
                q_values[q][i] = val
            counts[i] = yi.size

    x_centers = 0.5 * (bins[:-1] + bins[1:])

    if return_counts:
        return x_centers, q_values, counts
    else:
        return x_centers, q_values

In [ ]:
def n_ciddor(lam, t=20.0, p=101325.0, rh=20.0):
    """
    Refractive index of moist air via the Ciddor (1996) formula.

    Parameters
    ----------
    lam : array_like
        Wavelength in **microns**.
    t   : float
        Temperature in °C (default 20).
    p   : float
        Pressure in Pa (default 101 325).
    rh  : float
        Relative humidity in % (default 20).

    Returns
    -------
    n : ndarray
        Refractive index n(λ).
    """
    return ref_index.ciddor(wave=np.asarray(lam) * 1000.0, t=t, p=p, rh=rh)

In [ ]:
# Pre-compute n(λ) on the global grid once
N_LAM = n_ciddor(LAM)

In [ ]:
def compute_sigma_n(band, lam=LAM, n_lam=N_LAM):
    """
    Chromatic dispersion of the refractive index in an LSST band.

    The integration weight is the **photon-count weight** for a flat f_nu SED:

        w(lambda) = T_b(lambda) / lambda

    which comes from:
        dN_gamma/dlambda  ∝  T_b * f_lambda * lambda/(hc)
                          =  T_b * (f_nu/lambda^2) * lambda/(hc)
                          ∝  T_b / lambda          (for f_nu = const)

    The dispersion is the normalised weighted RMS:

        sigma_n(b) = sqrt( <(n - <n>_b)^2>_b )

    where <.>_b denotes the w(lambda)-weighted average over the band.

    Parameters
    ----------
    band  : str
        One of 'u', 'g', 'r', 'i', 'z', 'y'.
    lam   : array_like
        Wavelength grid in **microns** (default: global LAM grid, 0.3–1.1 µm).
    n_lam : array_like
        Refractive index evaluated on *lam* (default: N_LAM from Ciddor).
        Can be n(λ) or (n(λ)-1); only *variations* within the band matter.

    Returns
    -------
    sigma_n : float
        Photon-weighted RMS chromatic spread of n(λ) across the band
        (dimensionless).

    Notes
    -----
    To obtain the expected DCR dipole length in arcsec:
        l_dip [arcsec] = compute_sigma_n(band) * tan(z) * RAD_TO_ARCSEC
    """
    lam = np.asarray(lam)
    n_lam = np.asarray(n_lam)

    # --- Filter throughput interpolated onto the common wavelength grid -------
    bp = LSST_FILTERS[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4  # Å → µm
    T = np.interp(lam, lam_bp, bp.response, left=0.0, right=0.0)

    # --- Photon-count weight: w(λ) = T(λ) / λ  (flat f_nu SED) --------------
    w_raw = T / lam

    # --- Normalise so that integral(w, lam) = 1 ------------------------------
    norm = simpson(w_raw, lam)
    if norm == 0.0:
        return 0.0
    w = w_raw / norm

    # --- Band-averaged refractive index --------------------------------------
    n_mean = simpson(w * n_lam, lam)

    # --- Photon-weighted variance  →  sigma_n --------------------------------
    var_n = simpson(w * (n_lam - n_mean) ** 2, lam)
    return np.sqrt(var_n)

## 2. Observing-geometry helper

The parallactic angle and the hour angle are both computed from the LST:

$$H = \mathrm{LST} - \alpha \qquad\qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\cos\delta - \sin\delta\cos H\right)$$

The function returns η, H (hours and degrees), azimuth, zenith angle, sin z, and airmass.

### 2a. `zenith_tangent_vector` — tangent-plane projection (from obstime)

Projects the zenith direction into the tangent plane of the target using
the formula $\mathbf{v} = \mathbf{z} - (\mathbf{z}\cdot\mathbf{s})\,\mathbf{s}$,
where $\mathbf{s}$ is the unit vector toward the source and $\mathbf{z}$ is the
zenith unit vector obtained by transforming AltAz(alt=90°) to ICRS.
This version calls `astropy` for the time transform and is used for individual
sanity checks.

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith

    Parameters:
    ==========
        ra_deg,dec_deg: target coordinates in the sky
        obstimes:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
    """

    # Source
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Zénith en AltAz → (alt=90°, az arbitraire)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))

    # Convertir en ICRS
    zenith_icrs = zenith_altaz.transform_to("icrs")

    # Vecteurs cartésiens
    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    # Projection tangentielle
    v = z - np.dot(z, s) * s

    # Normalisation
    v /= np.linalg.norm(v)

    return v  # vecteur 3D tangent au ciel

### 2b. `zenith_tangent_vector_fromHA` — tangent-plane projection (from HA grid)

Same projection as above but computed analytically from a **precomputed hour-angle array** —
avoids repeated `astropy` time calls and is fast enough to sweep the full
$H\in[-180°,+180°]$ range for all DDFs.
The function also returns $\|\mathbf{v}\| = \sin z$, the zenith-angle sine
that governs DCR amplitude.

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

### 2c. `sinz_vs_HA` — zenith-angle sine from analytic formula

Direct analytic computation:
$$\sin z(H,\delta,\phi) = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$
Used to cross-check the norm returned by `zenith_tangent_vector_fromHA`
and to overlay the alert's measured $\sin z$ on the model curve.

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of sinz angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
def tanz_vs_HA(HA_deg, coords, location):
    """
    Compute the tan of zenith angle

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of tanz  angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    sinz = np.sqrt(1 - cosz**2)
    tanz = sinz / cosz
    return tanz

### 2d. `calculate_parallactic_angle` — η from obstime

Computes the parallactic angle from first principles given `astropy` `Time`
objects:
$$H = \mathrm{LST} - \alpha, \qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\,\cos\delta - \sin\delta\,\cos H\right)$$
Returns η in degrees, range $[-180°, +180°]$.

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle(coords, times, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # LST
    lst = times.sidereal_time("apparent", longitude=location.lon)

    # angle horaire H = LST - RA
    H = (lst - coords.ra).to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(H)
    cosH = np.cos(H)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

### 2e. `calculate_parallactic_angle_fromHA` — η from HA grid

Same formula as above but accepts a **precomputed hour-angle** `Angle` object
(in degrees) instead of `Time`.  Used to sweep the full HA range for
all DDFs without triggering IERS/UT1 network calls.

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        ha:  hour angle Angle in degree
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # angle horaire H = LST - RA
    ha_rad = ha.to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

### 2f. compute_observing_geometry — full geometry pipeline

Main workhorse: given arrays of (RA, Dec, MJD), returns a DataFrame with all derived observing-geometry columns (η, H, azimuth, altitude, zenith, , airmass). Processes alerts in batches of batch_size to keep memory usage bounded. A quick sanity check on COSMOS is run at the end.

In [ ]:
def compute_observing_geometry(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute full observing geometry for a set of alerts.

    Parameters
    ----------
    ra_deg, dec_deg : array-like – ICRS coordinates in degrees
    mjd             : array-like – MJD TAI
    location        : EarthLocation
    batch_size      : int – alerts per astropy call (speed/memory trade-off)

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg  float   −180 … +180°   (North = 0, CCW)
        hour_angle_hr          float   −12 … +12 h     (H = LST − RA)
        hour_angle_deg         float   −180 … +180°    (same × 15)
        azimuth_deg            float      0 … 360°    (North = 0, E = 90)
        altitude_deg           float      0 …  90°
        zenith_angle_deg       float      0 …  90°
        sin_zenith             float      0 … 1
        tan_zenith             float
        airmass                float   ≥ 1             (≈ 1/cos z)
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    H_hr = np.full(n, np.nan)  # hour angle in hours
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            # Sidereal time requires UT1
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)

            # LST and hour angle
            lst = times.sidereal_time("apparent", longitude=location.lon)
            H_wrap = (lst - coords.ra).wrap_at(180 * u.deg)  # Angle in (−180°, +180°]
            H_rad = H_wrap.to(u.rad).value
            H_hr[sl] = H_wrap.to(u.hourangle).value  # hours

            # Parallactic angle: η = arctan2(sin H, tan φ cos δ − sin δ cos H)
            phi = location.lat.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value
            para[sl] = np.degrees(
                np.arctan2(
                    np.sin(H_rad),
                    np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H_rad),
                )
            )

            # Alt/Az
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "hour_angle_hr": H_hr,
            "hour_angle_deg": H_hr * 15.0,  # 1 h = 15°
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "sin_zenith": np.sin(np.radians(za)),
            "tan_zenith": np.tan(np.radians(za)),
            "airmass": airmass,
        }
    )


# Sanity check
test = compute_observing_geometry([150.1191], [2.2058], [60310.5])
print("Sanity check COSMOS MJD=60310.5:")
print(test.to_string(index=False))

## Pre-Initlialisation

### a) Load LSST filters

In [ ]:
_lsst_seq = filters.load_filters("lsst2023-*")
LSST_FILTERS = {f.name: f for f in _lsst_seq}  # keyed as 'lsst2023-u', etc.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for band in BAND_ORDER:
    filter_name = f"lsst2023-{band}"
    throughput = LSST_FILTERS[filter_name]
    x = throughput.wavelength
    y = throughput.response
    ax.plot(x, y, color=BAND_COLORS[band], label=filter_name)
ax.legend(title="from speclite:", bbox_to_anchor=(1.1, 1.05))
ax.set_xlabel("$\lambda$ (nm)")
ax.set_title("LSSST throughput")
plt.show()

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"{'Band':>5}   {'sigma_n':>12}   {'l_dip(z=45°) [arcsec]':>22}")
print("-" * 46)
for b in BAND_ORDER:
    sn = compute_sigma_n(b)
    ldip_45 = sn * np.tan(np.deg2rad(45.0)) * RAD_TO_ARCSEC
    print(f"  {b}      {sn:.4e}          {ldip_45:.5f}")

### b) Pre-compute sigma_n for all bands

`compute_sigma_n` is purely spectro-photometric (no geometry), so we call it once
and cache the results in `SIGMA_N`.

In [ ]:
# sigma_n(b) for each LSST band — reusable in other notebooks
SIGMA_N = {b: compute_sigma_n(b) for b in BAND_ORDER}

print("sigma_n per band (photon-count weight, flat f_nu SED):")
for b, sn in SIGMA_N.items():
    print(f"  {b}:  {sn:.5e}")

## 3. Load dipole alerts

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DDF_NAMES:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean isDipole
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df_dip = (
        df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        if "r:isDipole" in df.columns
        else pd.DataFrame()
    )
    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip
    print(f"[{field_name:12s}] {len(df):7,} total  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 4. Compute observing geometry + derived quantities

New derived columns added here:

| Column | Definition |
|--------|------------|
| `sin_zenith` | sin(zenith_angle_deg) — proxy for DCR amplitude |
| `hour_angle_hr` | H = LST − RA in hours, wrapped to (−12h, +12h] |
| `hour_angle_deg` | same × 15, in degrees |
| `delta_dipole_para` | `r:dipoleAngle` − η, wrapped to (−180°, +180°] — signed difference |
| `delta_dipole_para_folded` | min(|Δ|, 180°−|Δ|) ∈ [0°, 90°] — headless (axis without direction) |

In [ ]:
frames: list[pd.DataFrame] = []

for field_name in DDF_NAMES:
    df_dip = ddf_alerts.get(field_name, pd.DataFrame())
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    need = ["r:ra", "r:dec", "r:midpointMjdTai"]
    missing = [c for c in need if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing {missing} — skipping.")
        continue

    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)
    print(f"[{field_name:12s}] computing geometry for {len(df_clean):,} dipoles …", end=" ")

    geo = compute_observing_geometry(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )
    df_clean = pd.concat([df_clean, geo], axis=1)

    # ── Angular differences between dipole angle and parallactic angle ──────
    if "r:dipoleAngle" in df_clean.columns:
        raw = df_clean["r:dipoleAngle"].values
        parang = df_clean["parallactic_angle_deg"].values

        # Signed difference wrapped to (−180°, +180°]
        diff = (raw - parang + 180.0) % 360.0 - 180.0
        df_clean["delta_dipole_para"] = diff

        # Headless (folded) difference: dipole axis has no preferred direction,
        # so 0° and 180° are equivalent.  Fold into [0°, 90°].
        # |Δ| mod 180 then fold to [0,90]
        adiff = np.abs(diff)  # [0, 180]
        folded = np.where(adiff <= 90.0, adiff, 180.0 - adiff)  # [0, 90]
        df_clean["delta_dipole_para_folded"] = folded

    frames.append(df_clean)
    print("done")

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal dipoles with geometry: {len(df_all):,}")
    cols_show = [
        "field",
        "r:band",
        "r:midpointMjdTai",
        "r:dipoleAngle",
        "r:dipoleLength",
        "parallactic_angle_deg",
        "hour_angle_hr",
        "hour_angle_deg",
        "zenith_angle_deg",
        "sin_zenith",
        "tan_zenith",
        "azimuth_deg",
        "altitude_deg",
        "airmass",
        "delta_dipole_para",
        "delta_dipole_para_folded",
    ]
    display(df_all[[c for c in cols_show if c in df_all.columns]].describe())
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

### Add prediction for dipole length

In [ ]:
df_all["dipole_length_pred"] = df_all["r:band"].map(SIGMA_N) * df_all["tan_zenith"] * RAD_TO_ARCSEC

## 5. Helper functions for rose diagrams and bar plots

### 5a. 360° rose diagram stacked by band (single axes)

In [ ]:
def rose_stacked_bands(
    ax,
    df_field: pd.DataFrame,
    angle_col: str,
    n_bins: int = 36,
    title: str = "",
    show_uniform: bool = True,
    angle_range: tuple = (0, 360),
) -> None:
    """
    Draw a rose diagram on a polar axes, stacking each band.

    Convention: North at top, clockwise (East to the right).
    The angle_range parameter controls the histogram domain;
    values are first wrapped to [angle_range[0], angle_range[1]).

    Parameters
    ----------
    ax           : matplotlib polar Axes
    df_field     : DataFrame for one DDF
    angle_col    : column name (degrees)
    n_bins       : number of azimuthal bins
    title        : subplot title
    show_uniform : draw the isotropic reference circle
    angle_range  : (lo, hi) in degrees – histogram domain
    """
    lo, hi = angle_range
    span = hi - lo
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    centers_d = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    centers = np.radians(centers_d)
    width = 2 * np.pi * (span / 360.0) / n_bins * 0.90

    bands_present = (
        [b for b in BAND_ORDER if b in df_field["r:band"].dropna().unique()]
        if "r:band" in df_field.columns
        else []
    )

    bottom = np.zeros(n_bins)
    for band in bands_present:
        vals = df_field.loc[df_field["r:band"] == band, angle_col].dropna().values
        # wrap to [lo, hi)
        vals = lo + (vals - lo) % span
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(
            centers,
            cnts,
            width=width,
            bottom=bottom,
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += cnts

    if show_uniform and bottom.sum() > 0:
        uniform = bottom.sum() / n_bins
        theta_ring = np.linspace(np.radians(lo), np.radians(hi), 360)
        ax.plot(
            theta_ring,
            np.full_like(theta_ring, uniform),
            "--",
            color="black",
            lw=0.8,
            alpha=0.6,
            label="uniform",
        )

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_thetalim(np.radians(lo), np.radians(hi))
    ax.tick_params(labelsize=7)
    ax.set_title(title, va="bottom", pad=14, fontsize=8)


print("rose_stacked_bands() defined.")

### 5b. 2×3 rose-diagram figure, one subplot per DDF

In [ ]:
def _band_legend_elements() -> list:
    elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    elems.append(plt.Line2D([0], [0], ls="--", color="black", lw=0.8, alpha=0.6, label="uniform"))
    return elems


def figure_rose_per_ddf(
    df_all: pd.DataFrame,
    angle_col: str,
    suptitle: str,
    figname: str,
    n_bins: int = 36,
    xlabel: str = "",
    angle_range: tuple = (0, 360),
) -> None:
    """
    2×3 figure of rose diagrams (one per DDF), stacked by band.

    Parameters
    ----------
    angle_range : (lo, hi) in degrees — for partial-circle roses (e.g. 0–180)
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 4.0, nrows * 4.2),
        subplot_kw={"projection": "polar"},
        layout="constrained",
    )

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        if sub.empty or angle_col not in sub.columns:
            ax.set_visible(False)
            continue
        n_total = sub[angle_col].notna().sum()
        rose_stacked_bands(
            ax,
            sub,
            angle_col,
            n_bins=n_bins,
            title=f"{field_name}  (n={n_total:,})",
            angle_range=angle_range,
        )

    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    ann = f"  [{xlabel}]" if xlabel else ""
    fig.suptitle(suptitle + ann, y=1.01, fontsize=11)
    fig.legend(
        handles=_band_legend_elements(),
        loc="lower center",
        ncol=8,
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.03),
    )
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_rose_per_ddf() defined.")

### 5c. 2×3 stacked bar-plot figure

In [ ]:
def figure_barplot_per_ddf(
    df_all: pd.DataFrame,
    value_col: str,
    suptitle: str,
    figname: str,
    n_bins: int = 20,
    xlabel: str = "",
    xlim: tuple = None,
) -> None:
    """2×3 stacked bar plots (one per DDF), one colour per band."""
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.5))

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        if sub.empty or value_col not in sub.columns:
            ax.set_visible(False)
            continue

        vals_all = sub[value_col].dropna().values
        vmin = xlim[0] if xlim else vals_all.min()
        vmax = xlim[1] if xlim else vals_all.max()
        bin_edges = np.linspace(vmin, vmax, n_bins + 1)
        centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
        width = (bin_edges[1] - bin_edges[0]) * 0.9

        bands_present = (
            [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
        )
        bottom = np.zeros(n_bins)
        for band in bands_present:
            vals_b = sub.loc[sub["r:band"] == band, value_col].dropna().values
            cnts, _ = np.histogram(vals_b, bins=bin_edges)
            ax.bar(
                centers,
                cnts,
                width=width,
                bottom=bottom,
                color=BAND_COLORS.get(band, "grey"),
                alpha=0.85,
                edgecolor="white",
                linewidth=0.3,
                label=band,
            )
            bottom += cnts

        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel("N dipoles", fontsize=8)
        ax.set_title(f"{field_name}  (n={sub[value_col].notna().sum():,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        ax.tick_params(labelsize=7)

    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS],
        loc="lower center",
        ncol=6,
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle(suptitle, y=1.01, fontsize=11)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_barplot_per_ddf() defined.")

### 5d. Scatter + median profile ± MAD (2×3 DDF + combined panel)

In [ ]:
def figure_scatter_profile(
    df_all: pd.DataFrame,
    x_col: str,
    y_col: str,
    suptitle: str,
    figname: str,
    xlabel: str = "",
    ylabel: str = "",
    n_bins: int = 15,
    xlim: tuple = None,
    ylim: tuple = None,
    abs_x: bool = False,
) -> None:
    """
    Draw a 2×3 subplot grid (one panel per DDF) PLUS one extra combined panel,
    laid out as a 3×3 grid with the combined panel in position (2, 2).

    Each panel: scatter plot coloured by band + median ± MAD profile.

    Parameters
    ----------
    abs_x : if True, use |x| for scatter and binning (e.g. |hour angle|)
    """
    # Layout: 3 rows × 3 cols; first 6 cells = DDFs, cell (2,2) = combined
    nrows, ncols = 3, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")

    # Hide unused cells (6 DDFs + 1 combined = 7 → cell 7 unused)
    axes[2][1].set_visible(False)

    def _draw_panel(ax, sub, title):
        if sub.empty:
            ax.set_visible(False)
            return
        mask = sub[x_col].notna() & sub[y_col].notna()
        sub = sub[mask]
        if len(sub) < 5:
            ax.set_visible(False)
            return

        xvals = np.abs(sub[x_col].values) if abs_x else sub[x_col].values
        yvals = sub[y_col].values

        # Scatter by band
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                idx_b = sub["r:band"] == band
                xb = np.abs(sub.loc[idx_b, x_col].values) if abs_x else sub.loc[idx_b, x_col].values
                ax.scatter(
                    xb,
                    sub.loc[idx_b, y_col].values,
                    s=10,
                    alpha=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    rasterized=True,
                    label=band,
                )
        else:
            ax.scatter(xvals, yvals, s=10, alpha=0.5, color="steelblue", rasterized=True)

        # Median ± MAD profile
        x_min = xlim[0] if xlim else xvals.min()
        x_max = xlim[1] if xlim else xvals.max()
        bin_edges = np.linspace(x_min, x_max, n_bins + 1)
        xc, med, mad = [], [], []
        for j in range(n_bins):
            sel = (xvals >= bin_edges[j]) & (xvals < bin_edges[j + 1])
            if sel.sum() < 3:
                continue
            ybin = yvals[sel]
            m = np.median(ybin)
            xc.append((bin_edges[j] + bin_edges[j + 1]) / 2)
            med.append(m)
            mad.append(np.median(np.abs(ybin - m)))
        if xc:
            xc = np.array(xc)
            med = np.array(med)
            mad = np.array(mad)
            ax.plot(xc, med, "k-", lw=1.5, label="median")
            ax.fill_between(xc, med - mad, med + mad, color="black", alpha=0.15, label="±MAD")

        # Spearman ρ on the displayed (possibly |x|) data
        r_s, _ = stats.spearmanr(xvals, yvals)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{title}  ρ={r_s:.3f}", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    # ── Per-DDF panels ────────────────────────────────────────────────────────
    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        _draw_panel(ax, sub, field_name)

    # ── Combined panel at position (2, 2) ─────────────────────────────────────
    ax_comb = axes[2][2]
    _draw_panel(ax_comb, df_all, "All DDFs combined")

    # Shared legend
    legend_elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    legend_elems += [
        plt.Line2D([0], [0], color="black", lw=1.5, label="median"),
        mpatches.Patch(facecolor="black", alpha=0.15, label="±MAD"),
    ]
    fig.legend(
        handles=legend_elems,
        loc="lower center",
        ncol=len(legend_elems),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )

    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_scatter_profile() defined.")

## 6. Rose diagrams — `r:dipoleAngle` (360°, stacked by band, 2×3 DDF)

In [ ]:
figure_rose_per_ddf(
    df_all,
    "r:dipoleAngle",
    suptitle="Rose diagram — r:dipoleAngle",
    figname="rose_dipoleAngle_per_ddf",
    xlabel="CCW from East (pixel axis); peaks at ±90° from N–S",
) if not df_all.empty else print("No data.")

## 7. Rose diagrams — Azimuth (360°, stacked by band, 2×3 DDF)

In [ ]:
figure_rose_per_ddf(
    df_all,
    "azimuth_deg",
    suptitle="Rose diagram — Azimuth",
    figname="rose_azimuth_per_ddf",
    xlabel="North=0°, East=+90° (CW, astropy standard)",
) if not df_all.empty else print("No data.")

## 8. Rose diagrams — Hour angle  (stacked by band, 2×3 DDF)

In [ ]:
figure_rose_per_ddf(
    df_all,
    "hour_angle_deg",
    suptitle="Rose diagram — Hour angle",
    figname="rose_hour angle_per_ddf",
    xlabel="North=0°",
) if not df_all.empty else print("No data.")

## 9. Rose diagrams — Parallactic angle η (360°, stacked by band, 2×3 DDF)

In [ ]:
figure_rose_per_ddf(
    df_all,
    "parallactic_angle_deg",
    suptitle="Rose diagram — Parallactic angle η",
    figname="rose_parallactic_per_ddf",
    xlabel="North=0°, CCW; identical formula to astroplan",
) if not df_all.empty else print("No data.")

## 10. Rose diagrams — Δ signed = `r:dipoleAngle` − η  (360°, stacked by band, 2×3 DDF)

The signed difference Δ ∈ (−180°, +180°] is wrapped back to [0°, 360°) for the rose diagram
so that North (0°) is at the top.  If the dipole axis perfectly tracks η, the distribution
clusters near 0° **and** near ±180° (because the dipole orientation has a 180° ambiguity).

In [ ]:
figure_rose_per_ddf(
    df_all,
    "delta_dipole_para",
    suptitle="Rose diagram — Δ signed = r:dipoleAngle − η  (wrapped to [0°, 360°))",
    figname="rose_delta_signed_per_ddf",
    xlabel="0° = perfect alignment; ±180° = anti-alignment (same axis)",
) if not df_all.empty else print("No data.")

## 10. Rose diagrams — Δ folded = |dipoleAngle − η| folded to [0°, 90°]  (2×3 DDF)

The dipole is a **headless** axis: 0° and 180° describe the same orientation.
Folding first collapses 180° → 0°, then maps (90°, 180°) → (90°, 0°), giving a
range [0°, 90°].  Perfect alignment piles up near 0°; perpendicular alignment near 90°.

The rose is plotted over 360° (the [0°, 90°] range is mirrored three times for visual clarity),
but the underlying histogram covers [0°, 360°) with the folded values repeated.

In [ ]:
# For visual clarity we plot the folded [0,90] histogram as a 360° rose
# by using n_bins=18 over [0,360] and mapping each value to all four quadrants.
# Simpler: just plot [0°, 90°] directly with angle_range.
figure_rose_per_ddf(
    df_all,
    "delta_dipole_para_folded",
    suptitle="Rose diagram — Δ folded = |dipoleAngle − η| folded to [0°, 90°]",
    figname="rose_delta_folded_per_ddf",
    n_bins=18,
    angle_range=(0, 90),
    xlabel="0° = aligned; 90° = perpendicular",
) if not df_all.empty else print("No data.")

## 11. Bar plots — Zenith angle distribution (stacked by band, 2×3 DDF)

In [ ]:
figure_barplot_per_ddf(
    df_all,
    "zenith_angle_deg",
    suptitle="Zenith angle distribution — stacked by band",
    figname="barplot_zenith_per_ddf",
    n_bins=20,
    xlabel="Zenith angle z (deg)",
    xlim=(0, 70),
) if not df_all.empty else print("No data.")

## 12. Bar plots — Airmass distribution (stacked by band, 2×3 DDF)

In [ ]:
figure_barplot_per_ddf(
    df_all,
    "airmass",
    suptitle="Airmass distribution — stacked by band",
    figname="barplot_airmass_per_ddf",
    n_bins=20,
    xlabel="Airmass X  (≈ 1/cos z)",
    xlim=(1.0, 2.3),
) if not df_all.empty else print("No data.")

## 13. `r:dipoleLength` vs sin(z) — scatter + median ± MAD

**Physical motivation.**  Differential chromatic refraction shifts a source by
an amount proportional to $\tan z \approx \sin z$ (for $z < 60°$).  If the dipole
is produced by template–science PSF mismatch driven by DCR, the separation
`r:dipoleLength` should grow linearly with $\sin z$.

The median ± MAD profile makes the trend visible despite the large scatter
from other sources of variability (seeing, focal-plane position, …).

In [ ]:
if not df_all.empty and "r:dipoleLength" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="tan_zenith",
        y_col="r:dipoleLength",
        suptitle="r:dipoleLength vs tan(zenith angle)",
        figname="scatter_dipoleLength_vs_tanz",
        xlabel="tan z",
        ylabel="r:dipoleLength (arcsec)",
        n_bins=20,
        xlim=(0.0, 2.0),
        ylim=(0.0, 0.3),
    )
else:
    print("r:dipoleLength or sin_zenith missing — skipping.")

## 14. `r:dipoleAngle` vs tan(z) — scatter + median ± MAD

The **angle** of the dipole should not depend on zenith angle (only the parallactic angle
does), but this plot is a useful sanity check.  A flat median profile confirms the absence
of spurious correlations with sin z.

In [ ]:
if not df_all.empty and "r:dipoleAngle" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="tan_zenith",
        y_col="r:dipoleAngle",
        suptitle="r:dipoleAngle vs tan(zenith angle)  — expected: flat median",
        figname="scatter_dipoleAngle_vs_sinz",
        xlabel="tan z",
        ylabel="r:dipoleAngle (deg)",
        n_bins=15,
        xlim=(0.0, 2.0),
    )
else:
    print("No data — skipping.")

## 15. `r:dipoleLength` vs |Hour angle| — scatter + median ± MAD

**Physical motivation.**  At transit (H = 0) the parallactic angle changes fastest;
at large |H| the angle stabilises.  The *amplitude* of DCR depends on zenith angle z,
which is related to H, δ, and φ by

$$\cos z = \sin\phi\sin\delta + \cos\phi\cos\delta\cos H$$

Plotting `r:dipoleLength` vs |H| probes this geometric dependence per DDF
(each DDF has a unique δ, so the z(H) curve differs between fields).

We use |H| because the dipole amplitude is symmetric in hour angle.

In [ ]:
if not df_all.empty and "r:dipoleLength" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleLength",
        suptitle="r:dipoleLength vs |Hour angle|  (per DDF)",
        figname="scatter_dipoleLength_vs_absH",
        xlabel="|H|  (hours)",
        ylabel="r:dipoleLength (pixels)",
        n_bins=15,
        xlim=(0, 6),
        ylim=(0, 0.3),
        abs_x=True,
    )
else:
    print("No data — skipping.")

## 16. `r:dipoleAngle` vs Hour angle (signed) — scatter + median ± MAD

The parallactic angle η is a monotone function of the signed hour angle H.
Since we have established that `r:dipoleAngle` ≈ η (mod 180°), we expect
a smooth S-shaped relationship between `r:dipoleAngle` and H.

This figure uses the **signed** H (not the absolute value) so that the S-curve
is visible.  Each DDF has a different shape because the S-curve depends on δ and φ.

In [ ]:
if not df_all.empty and "r:dipoleAngle" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleAngle",
        suptitle="r:dipoleAngle vs Hour angle H (signed)",
        figname="scatter_dipoleAngle_vs_Hr",
        xlabel="H  (hours)",
        ylabel="r:dipoleAngle (deg)",
        n_bins=20,
        xlim=(-6, 6),
        abs_x=False,
    )
else:
    print("No data — skipping.")

## 17. `r:dipoleAngle` vs Hour angle in degrees — scatter + median ± MAD

Same as §16 with H expressed in degrees (H_deg = H_hr × 15) for easier
comparison with the parallactic-angle rose diagrams.

In [ ]:
if not df_all.empty and "r:dipoleAngle" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="hour_angle_deg",
        y_col="r:dipoleAngle",
        suptitle="r:dipoleAngle vs Hour angle H in degrees",
        figname="scatter_dipoleAngle_vs_Hdeg",
        xlabel="H  (degrees)",
        ylabel="r:dipoleAngle (deg)",
        n_bins=24,
        xlim=(-100, 100),
        abs_x=False,
    )
else:
    print("No data — skipping.")

## 18. Parallactic angle η vs Hour angle H — scatter + median ± MAD

Cross-check: η(H) should follow a smooth S-curve, symmetric around H=0,
with amplitude that depends on δ and φ.  This panel confirms the geometry
is correctly computed and provides a visual reference for §16.

In [ ]:
if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="hour_angle_deg",
        y_col="parallactic_angle_deg",
        suptitle="Parallactic angle η vs Hour angle H  — S-curve cross-check",
        figname="scatter_parallactic_vs_H",
        xlabel="H  (deg)",
        ylabel="Parallactic angle η (deg)",
        n_bins=24,
        xlim=(-180, 180),
        abs_x=False,
    )
else:
    print("No data — skipping.")

In [ ]:
if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_scatter_profile(
        df_all,
        x_col="hour_angle_hr",
        y_col="parallactic_angle_deg",
        suptitle="Parallactic angle η vs Hour angle H  — S-curve cross-check",
        figname="scatter_parallactic_vs_H",
        xlabel="H  (hours)",
        ylabel="Parallactic angle η (deg)",
        n_bins=24,
        xlim=(-12, 12),
        abs_x=False,
    )
else:
    print("No data — skipping.")

In [ ]:
def figure_scatter_with_theory(
    df_all: pd.DataFrame,
    x_col: str,  # "hour_angle_deg" or "hour_angle_hr"
    y_col: str,  # "parallactic_angle_deg" or "sin_zenith"
    theory_func,  # callable(H_deg_array, ra_deg, dec_deg) -> y_theory_array
    suptitle: str,
    figname: str,
    xlabel: str = "",
    ylabel: str = "",
    xlim: tuple = (-180, 180),
    ylim: tuple = None,
    x_is_hours: bool = False,  # if True x_col is in hours; theory grid in hours
) -> None:
    """
    2x3 subplot grid (one per DDF).
    Each panel: scatter by band + theoretical curve computed from first principles.

    Parameters
    ----------
    theory_func : callable(H_deg_1d, ra_deg_scalar, dec_deg_scalar) -> y_theory_1d
        Must accept a 1-D numpy array of H values IN DEGREES and the scalar
        (RA, Dec) of the DDF, and return the theoretical y values.
    x_is_hours : bool
        If True the scatter x-axis is in hours (xlim should be +-12).
        The theory curve is still computed in degrees then converted.
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")

    # Dense H grid for the theory curve (always in degrees internally)
    H_theory_deg = np.linspace(-180, 180, 720)

    def _draw_panel(ax, sub, field_name, ra_deg, dec_deg):
        if sub.empty:
            ax.set_visible(False)
            return
        mask = sub[x_col].notna() & sub[y_col].notna()
        sub = sub[mask]
        if len(sub) < 3:
            ax.set_visible(False)
            return

        # scatter by band
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                idx_b = sub["r:band"] == band
                ax.scatter(
                    sub.loc[idx_b, x_col].values,
                    sub.loc[idx_b, y_col].values,
                    s=20,
                    alpha=0.2,
                    color=BAND_COLORS.get(band, "grey"),
                    rasterized=True,
                    label=band,
                )
        else:
            ax.scatter(
                sub[x_col].values, sub[y_col].values, s=10, alpha=0.5, color="steelblue", rasterized=True
            )

        # theory curve
        y_theory = theory_func(H_theory_deg, ra_deg, dec_deg)
        x_theory = H_theory_deg / 15.0 if x_is_hours else H_theory_deg
        ax.plot(x_theory, y_theory, "k-", lw=1.5, zorder=5, label="theory")

        n_pts = mask.sum()
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{field_name}  (n={n_pts:,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        ra_deg, dec_deg = DEEP_FIELDS[field_name]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        _draw_panel(ax, sub, field_name, ra_deg, dec_deg)

    # Shared legend
    legend_elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    legend_elems.append(plt.Line2D([0], [0], color="black", lw=1.5, label="theory"))
    fig.legend(
        handles=legend_elems,
        loc="lower center",
        ncol=len(legend_elems),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()


# Theory functions


def figure_scatter_with_theory_perband(
    df_all: pd.DataFrame,
    x_col: str,  # "hour_angle_deg" or "hour_angle_hr"
    y_col: str,  # "parallactic_angle_deg" or "sin_zenith"
    theory_func,  # callable(H_deg_array, ra_deg, dec_deg) -> y_theory_array
    suptitle: str,
    figname: str,
    xlabel: str = "",
    ylabel: str = "",
    xlim: tuple = (-180, 180),
    ylim: tuple = None,
    x_is_hours: bool = False,  # if True x_col is in hours; theory grid in hours
) -> None:
    """
    2x3 subplot grid (one per DDF).
    Each panel: scatter by band + theoretical curve computed from first principles.
    ==> Note the theory is only related to dipole length
    Parameters
    ----------
    theory_func : callable(H_deg_1d, ra_deg_scalar, dec_deg_scalar) -> y_theory_1d
        Must accept a 1-D numpy array of H values IN DEGREES and the scalar
        (RA, Dec) of the DDF, and return the theoretical y values.
    x_is_hours : bool
        If True the scatter x-axis is in hours (xlim should be +-12).
        The theory curve is still computed in degrees then converted.
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")

    # Dense H grid for the theory curve (always in degrees internally)
    H_theory_deg = np.linspace(-180, 180, 720)

    def _draw_panel(ax, sub, field_name, ra_deg, dec_deg):
        if sub.empty:
            ax.set_visible(False)
            return
        mask = sub[x_col].notna() & sub[y_col].notna()
        sub = sub[mask]
        if len(sub) < 3:
            ax.set_visible(False)
            return

        # scatter by band
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                idx_b = sub["r:band"] == band
                ax.scatter(
                    sub.loc[idx_b, x_col].values,
                    sub.loc[idx_b, y_col].values,
                    s=20,
                    alpha=0.2,
                    color=BAND_COLORS.get(band, "grey"),
                    rasterized=True,
                    label=band,
                )
        else:
            ax.scatter(
                sub[x_col].values, sub[y_col].values, s=10, alpha=0.5, color="steelblue", rasterized=True
            )

        # theory curve
        # y_theory = theory_func(H_theory_deg, ra_deg, dec_deg)
        # x_theory = H_theory_deg / 15.0 if x_is_hours else H_theory_deg
        # ax.plot(x_theory, y_theory, "k-", lw=1.5, zorder=5, label="theory")

        # show the theory
        for b in BAND_ORDER:
            color = BAND_COLORS.get(b, "grey")
            # theory curve by computing the dipole length
            y_theory = SIGMA_N[b] * theory_func(H_theory_deg, ra_deg, dec_deg) * RAD_TO_ARCSEC
            x_theory = H_theory_deg / 15.0 if x_is_hours else H_theory_deg
            ax.plot(x_theory, y_theory, "k-", lw=1.0, color=color, zorder=5, label=f"{b} (th)")

        n_pts = mask.sum()
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{field_name}  (n={n_pts:,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        ra_deg, dec_deg = DEEP_FIELDS[field_name]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        _draw_panel(ax, sub, field_name, ra_deg, dec_deg)

    # Shared legend
    legend_elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    legend_elems.append(plt.Line2D([0], [0], color="black", lw=1.5, label="theory"))
    fig.legend(
        handles=legend_elems,
        loc="lower center",
        ncol=len(legend_elems),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()


# Theory functions


def theory_parallactic(H_deg, ra_deg, dec_deg):
    """eta(H) = arctan2(sin H, tan phi cos delta - sin delta cos H)  [degrees]"""
    H = np.deg2rad(H_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)
    return np.degrees(np.arctan2(np.sin(H), np.tan(phi) * np.cos(dec) - np.sin(dec) * np.cos(H)))


def theory_sinz(H_deg, ra_deg, dec_deg):
    """sin z(H) = sqrt(1 - (sin phi sin delta + cos phi cos delta cos H)^2)"""
    H = np.deg2rad(H_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)
    cosz = np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.cos(H)
    return np.sqrt(np.maximum(0.0, 1.0 - cosz**2))


def theory_tanz(H_deg, ra_deg, dec_deg):
    """sin z(H) = sqrt(1 - (sin phi sin delta + cos phi cos delta cos H)^2)"""
    H = np.deg2rad(H_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)
    cosz = np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.cos(H)
    sinz = np.sqrt(np.maximum(0.0, 1.0 - cosz**2))
    tanz = sinz / cosz
    return tanz


print("figure_scatter_with_theory(), theory_parallactic(), theory_sinz() defined.")

### 18b. eta(H) with theoretical S-curve (2x3 per DDF)

Scatter of observed parallactic angle eta vs hour angle H, coloured by band,
overlaid with the theoretical curve

$$\eta(H)=\arctan2\!\left(\sin H,\;\tan\phi\cos\delta-\sin\delta\cos H\right)$$

computed for the exact (RA, Dec) of each DDF and the Rubin latitude phi = -30.24 deg.
The data points should fall **exactly** on the curve since eta was derived from the
same formula.  The plot shows which portion of the full H range is actually sampled.


In [ ]:
if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_scatter_with_theory(
        df_all,
        x_col="hour_angle_deg",
        y_col="parallactic_angle_deg",
        theory_func=theory_parallactic,
        suptitle="Parallactic angle eta vs Hour angle H (deg) — scatter + theory",
        figname="scatter_theory_parallactic_vs_Hdeg",
        xlabel="H  (deg)",
        ylabel="Parallactic angle eta (deg)",
        xlim=(-180, 180),
        ylim=(-180, 180),
        x_is_hours=False,
    )
else:
    print("No data — skipping.")

In [ ]:
if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_scatter_with_theory(
        df_all,
        x_col="hour_angle_hr",
        y_col="parallactic_angle_deg",
        theory_func=theory_parallactic,
        suptitle="Parallactic angle eta vs Hour angle H (hours) — scatter + theory",
        figname="scatter_theory_parallactic_vs_Hhr",
        xlabel="H  (hours)",
        ylabel="Parallactic angle eta (deg)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

In [ ]:
if not df_all.empty and "r:dipoleAngle" in df_all.columns:
    figure_scatter_with_theory(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleAngle",
        theory_func=theory_parallactic,
        suptitle="dipole angle  vs Hour angle H (hours) — scatter + theory",
        figname="dipoleangle_meas_parallactic_vs_Hr",
        xlabel="H  (hours)",
        ylabel="dipole angle eta (deg)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

In [ ]:
df_cosmos = df_all[df_all["field"] == "COSMOS"]

In [ ]:
df_cosmos.columns

### Test profile histogram

In [ ]:
nbinsHA = 30
binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)
xv = df_cosmos["hour_angle_hr"].values
yv = df_cosmos["r:dipoleAngle"].values

x_centers, y_med, y_err, y_rms = profile_median(xv, yv, binsHAhour)
x_centers, q_values = profile_quantile(xv, yv, binsHAhour, quantiles=(0.05, 0.5, 0.95))

In [ ]:
fig = plt.figure()
plt.errorbar(x_centers, y_med, yerr=y_err, fmt="o", capsize=3)
plt.xlabel("X")
plt.ylabel("Median Y")
plt.show()

### Compare Data wrt theory without color(band) dependence 

In [ ]:
def figure_profilehist_with_theory(
    df_all: pd.DataFrame,
    x_col: str,  # "hour_angle_deg" or "hour_angle_hr"
    y_col: str,  # "parallactic_angle_deg" or "sin_zenith"
    x_col_bins: npt.NDArray[np.float64],  # binning of X axis
    theory_func,  # callable(H_deg_array, ra_deg, dec_deg) -> y_theory_array
    suptitle: str,
    figname: str,
    xlabel: str = "",
    ylabel: str = "",
    xlim: tuple = (-180, 180),
    ylim: tuple = None,
    x_is_hours: bool = False,  # if True x_col is in hours; theory grid in hours
) -> None:
    """
    2x3 subplot grid (one per DDF).
    Each panel: scatter by band + theoretical curve computed from first principles.

    Parameters
    ----------
    theory_func : callable(H_deg_1d, ra_deg_scalar, dec_deg_scalar) -> y_theory_1d
        Must accept a 1-D numpy array of H values IN DEGREES and the scalar
        (RA, Dec) of the DDF, and return the theoretical y values.
    x_is_hours : bool
        If True the profile x-axis is in hours (xlim should be +-12).
        The theory curve is still computed in degrees then converted.
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")

    # Dense H grid for the theory curve (always in degrees internally)
    H_theory_deg = np.linspace(-180, 180, 720)

    def _draw_panel(ax, sub, field_name, ra_deg, dec_deg):
        if sub.empty:
            ax.set_visible(False)
            return
        mask = sub[x_col].notna() & sub[y_col].notna()
        sub = sub[mask]
        if len(sub) < 3:
            ax.set_visible(False)
            return

        # scatter by band
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                idx_b = sub["r:band"] == band
                # xv = sub.loc[idx_b, x_col].values
                # yy = sub.loc[idx_b, y_col].values

                xv = np.asarray(sub.loc[idx_b, x_col]).ravel()
                yv = np.asarray(sub.loc[idx_b, y_col]).ravel()

                x_centers, y_med, y_err, y_rms = profile_median(xv, yv, x_col_bins)
                color = BAND_COLORS.get(band, "grey")
                ax.errorbar(x_centers, y_med, yerr=y_err, fmt="o", capsize=3, color=color, label=band)
        else:
            xv = sub[x_col].values
            yy = sub[y_col].values
            x_centers, y_med, y_err, y_rms = profile_median(xv, yv, x_col_bins)
            color = "k"
            ax.errorbar(x_centers, y_med, yerr=y_err, fmt="o", capsize=1, color=color)

        # theory curve
        y_theory = theory_func(H_theory_deg, ra_deg, dec_deg)
        x_theory = H_theory_deg / 15.0 if x_is_hours else H_theory_deg
        ax.plot(x_theory, y_theory, "k-", lw=1.5, zorder=5, label="theory")

        n_pts = mask.sum()
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{field_name}  (n={n_pts:,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)
        ax.legend()

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        ra_deg, dec_deg = DEEP_FIELDS[field_name]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        _draw_panel(ax, sub, field_name, ra_deg, dec_deg)

    # Shared legend
    legend_elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    legend_elems.append(plt.Line2D([0], [0], color="black", lw=1.5, label="theory"))
    fig.legend(
        handles=legend_elems,
        loc="lower center",
        ncol=len(legend_elems),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()

### Compare Data wrt theory with color(band) dependence 

In [ ]:
def figure_profilehist_with_theory_perband(
    df_all: pd.DataFrame,
    x_col: str,  # "hour_angle_deg" or "hour_angle_hr"
    y_col: str,  # "parallactic_angle_deg" or "sin_zenith"
    x_col_bins: npt.NDArray[np.float64],  # binning of X axis
    theory_func,  # callable(H_deg_array, ra_deg, dec_deg) -> y_theory_array
    suptitle: str,
    figname: str,
    xlabel: str = "",
    ylabel: str = "",
    xlim: tuple = (-180, 180),
    ylim: tuple = None,
    x_is_hours: bool = False,  # if True x_col is in hours; theory grid in hours
) -> None:
    """
    2x3 subplot grid (one per DDF).
    Each panel: scatter by band + theoretical curve computed from first principles.
      ==> Note the theory is only related to dipole length
    Parameters
    ----------
    theory_func : callable(H_deg_1d, ra_deg_scalar, dec_deg_scalar) -> y_theory_1d
        Must accept a 1-D numpy array of H values IN DEGREES and the scalar
        (RA, Dec) of the DDF, and return the theoretical y values.
    x_is_hours : bool
        If True the profile x-axis is in hours (xlim should be +-12).
        The theory curve is still computed in degrees then converted.
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")

    # Dense H grid for the theory curve (always in degrees internally)
    H_theory_deg = np.linspace(-180, 180, 720)

    def _draw_panel(ax, sub, field_name, ra_deg, dec_deg):
        if sub.empty:
            ax.set_visible(False)
            return
        mask = sub[x_col].notna() & sub[y_col].notna()
        sub = sub[mask]
        if len(sub) < 3:
            ax.set_visible(False)
            return

        # scatter by band
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                idx_b = sub["r:band"] == band
                # xv = sub.loc[idx_b, x_col].values
                # yy = sub.loc[idx_b, y_col].values

                xv = np.asarray(sub.loc[idx_b, x_col]).ravel()
                yv = np.asarray(sub.loc[idx_b, y_col]).ravel()

                x_centers, y_med, y_err, y_rms = profile_median(xv, yv, x_col_bins)
                color = BAND_COLORS.get(band, "grey")
                ax.errorbar(x_centers, y_med, yerr=y_err, fmt="o", capsize=3, color=color, label=band)
        else:
            xv = sub[x_col].values
            yy = sub[y_col].values
            x_centers, y_med, y_err, y_rms = profile_median(xv, yv, x_col_bins)
            color = "k"
            ax.errorbar(x_centers, y_med, yerr=y_err, fmt="o", capsize=1, color=color)

        # show the theory
        for b in BAND_ORDER:
            color = BAND_COLORS.get(b, "grey")
            # theory curve by computing the dipole length
            y_theory = SIGMA_N[b] * theory_func(H_theory_deg, ra_deg, dec_deg) * RAD_TO_ARCSEC
            x_theory = H_theory_deg / 15.0 if x_is_hours else H_theory_deg
            ax.plot(x_theory, y_theory, "k-", lw=1.0, color=color, zorder=5, label=f"{b} (th)")

        n_pts = mask.sum()
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{field_name}  (n={n_pts:,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        ra_deg, dec_deg = DEEP_FIELDS[field_name]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
        _draw_panel(ax, sub, field_name, ra_deg, dec_deg)

    # Shared legend
    # legend_elems = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    # legend_elems.append(plt.Line2D([0], [0], color="black", lw=1.5, label="theory"))
    # fig.legend(
    #    handles=legend_elems,
    #    loc="lower center",
    #    ncol=len(legend_elems),
    #    fontsize=8,
    #    frameon=False,
    #    bbox_to_anchor=(0.5, -0.04),
    # )

    legend_elems = []

    # points (data)
    for b in BAND_ORDER:
        if b in BAND_COLORS:
            legend_elems.append(
                plt.Line2D([0], [0], marker="o", linestyle="None", color=BAND_COLORS[b], label=b)
            )

        # lignes (théorie)
        legend_elems.append(plt.Line2D([0], [0], linestyle="-", color="black", label="theory"))

    fig.legend(
        handles=legend_elems,
        loc="lower center",
        ncol=len(legend_elems),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()

### Show Dipole angle wrt Hour angle and compate it with paralactic angle 

In [ ]:
nbinsHA = 50
# binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)

if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_profilehist_with_theory(
        df_all,
        x_col="hour_angle_deg",
        y_col="r:dipoleAngle",
        x_col_bins=binsHAdeg,
        theory_func=theory_parallactic,
        suptitle="Dipole angle vs Hour angle H (deg) — profile + theory",
        figname="profile_theory_parallactic_vs_Hdeg",
        xlabel="H (deg)",
        ylabel="Dipole angle (deg)",
        xlim=(-120, 120),
        ylim=(-180, 180),
        x_is_hours=False,
    )
else:
    print("No data — skipping.")

In [ ]:
nbinsHA = 50
binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
# binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)

if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_profilehist_with_theory(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleAngle",
        x_col_bins=binsHAhour,
        theory_func=theory_parallactic,
        suptitle="Dipole angle vs Hour angle H (hour) — profile + theory",
        figname="profile_theory_parallactic_vs_Hhr",
        xlabel="H (hour)",
        ylabel="Dipole angle (deg)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

### 18c. tan z vs H with theoretical curve (2x3 per DDF)

Scatter of observed sin(zenith angle) vs hour angle H, overlaid with the
theoretical curve

$$\sin z(H)=\sqrt{1-\left(\sin\phi\sin\delta+\cos\phi\cos\delta\cos H\right)^2}$$

The minimum of sin z (transit, H = 0) and the range of H actually sampled are
immediately visible.  For typical DDFs, |H| < 3 h, confirming that Rubin observes
near the meridian.


In [ ]:
if not df_all.empty and "tan_zenith" in df_all.columns:
    figure_scatter_with_theory(
        df_all,
        x_col="hour_angle_deg",
        y_col="tan_zenith",
        theory_func=theory_tanz,
        suptitle="tan(zenith angle) vs Hour angle H (deg) — scatter + theory",
        figname="scatter_theory_tanz_vs_Hdeg",
        xlabel="H (deg)",
        ylabel="tan z",
        xlim=(-90, 90),
        ylim=(0, 5),
        x_is_hours=False,
    )
else:
    print("No data — skipping.")

In [ ]:
if not df_all.empty and "tan_zenith" in df_all.columns:
    figure_scatter_with_theory(
        df_all,
        x_col="hour_angle_hr",
        y_col="tan_zenith",
        theory_func=theory_tanz,
        suptitle="tan(zenith angle) vs Hour angle H (hours) — scatter + theory",
        figname="scatter_theory_tanz_vs_Hhr",
        xlabel="H  (hours)",
        ylabel="tan z",
        xlim=(-6, 6),
        ylim=(0, 5),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

In [ ]:
nbinsHA = 50
# binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)

if not df_all.empty:
    figure_profilehist_with_theory_perband(
        df_all,
        x_col="hour_angle_deg",
        y_col="r:dipoleLength",
        x_col_bins=binsHAdeg,
        theory_func=theory_tanz,
        suptitle="Dipole Length vs Hour angle H (deg) — profile + theory",
        figname="profile_theory_dipolelength_vs_Hdeg_perband",
        xlabel="H  (deg)",
        ylabel="Dipole length (arcsec)",
        xlim=(-120, 120),
        ylim=(0, 0.15),
        x_is_hours=False,
    )
else:
    print("No data — skipping.")

In [ ]:
nbinsHA = 50
binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
# binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)

if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_profilehist_with_theory_perband(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleLength",
        x_col_bins=binsHAhour,
        theory_func=theory_tanz,
        suptitle="Dipole length vs Hour angle H (hour) — profile + theory",
        figname="profile_theory_tanz_vs_Hr_perband",
        xlabel="H (hour)",
        ylabel="Dipole length (arcsec)",
        xlim=(-6, 6),
        ylim=(0, 0.15),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

In [ ]:
nbinsHA = 50
binsHAhour = np.linspace(-6, 6, nbinsHA + 1)
# binsHAdeg = np.linspace(-120, 120, nbinsHA + 1)

if not df_all.empty and "parallactic_angle_deg" in df_all.columns:
    figure_scatter_with_theory_perband(
        df_all,
        x_col="hour_angle_hr",
        y_col="r:dipoleLength",
        theory_func=theory_tanz,
        suptitle="Dipole length vs Hour angle H (hour) — profile + theory",
        figname="scatter_theory_tanz_vs_Hr_perband",
        xlabel="H (hour)",
        ylabel="Dipole length (arcsec)",
        xlim=(-6, 6),
        ylim=(0, 0.15),
        x_is_hours=True,
    )
else:
    print("No data — skipping.")

## 19. Scatter + 2D histogram — `r:dipoleAngle` vs η (per DDF)

In [ ]:
if df_all.empty:
    print("No data — skipping.")
else:
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8))
    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name]
        mask = sub["r:dipoleAngle"].notna() & sub["parallactic_angle_deg"].notna()
        sub = sub[mask]
        if len(sub) < 5:
            ax.set_visible(False)
            continue
        x, y = sub["parallactic_angle_deg"].values, sub["r:dipoleAngle"].values
        h, xe, ye = np.histogram2d(x, y, bins=36)
        ax.pcolormesh((xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2, h.T, cmap="hot_r")
        r_s, _ = stats.spearmanr(x, y)
        ax.set_xlabel("Parallactic angle η (deg)", fontsize=8)
        ax.set_ylabel("r:dipoleAngle (deg)", fontsize=8)
        ax.set_title(f"{field_name}  ρ={r_s:.3f}", fontsize=8)
    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)
    fig.suptitle("2D histogram — r:dipoleAngle vs η", y=1.01, fontsize=11)
    plt.tight_layout()
    savefig("hist2d_dipoleAngle_vs_parallactic")
    plt.show()

## 20. Correlation summary table (Pearson & Spearman)

In [ ]:
x_pairs = [
    ("parallactic_angle_deg", "parallactic"),
    ("hour_angle_hr", "H_hr"),
    ("hour_angle_deg", "H_deg"),
    ("azimuth_deg", "azimuth"),
    ("zenith_angle_deg", "zenith"),
    ("tan_zenith", "tan_z"),
    ("airmass", "airmass"),
]
y_pairs = [
    ("r:dipoleAngle", "dipoleAngle"),
    ("r:dipoleLength", "dipoleLength"),
    #    ("delta_dipole_para", "delta_signed"),
    #    ("delta_dipole_para_folded", "delta_folded"),
]

rows = []
for field_name in DDF_NAMES:
    sub_f = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub_f.empty:
        continue
    bands_present = (
        [b for b in BAND_ORDER if b in sub_f["r:band"].dropna().unique()]
        if "r:band" in sub_f.columns
        else ["all"]
    )
    for band in bands_present:
        sub = sub_f[sub_f["r:band"] == band] if band != "all" else sub_f
        for x_col, x_lbl in x_pairs:
            for y_col, y_lbl in y_pairs:
                if x_col not in sub.columns or y_col not in sub.columns:
                    continue
                mask = sub[x_col].notna() & sub[y_col].notna()
                x = sub.loc[mask, x_col].values
                y = sub.loc[mask, y_col].values
                if len(x) < 5:
                    continue
                r_p, p_p = stats.pearsonr(x, y)
                r_s, p_s = stats.spearmanr(x, y)
                rows.append(
                    {
                        "field": field_name,
                        "band": band,
                        "x": x_lbl,
                        "y": y_lbl,
                        "n": int(mask.sum()),
                        "pearson_r": round(r_p, 4),
                        "pearson_p": round(p_p, 4),
                        "spearman_r": round(r_s, 4),
                        "spearman_p": round(p_s, 4),
                    }
                )

df_corr = pd.DataFrame(rows)
if not df_corr.empty:
    pd.set_option("display.max_rows", 300)
    display(df_corr.sort_values(["field", "band", "x", "y"]))
else:
    print("No correlation data available.")

## 21. Spearman ρ heatmaps

In [ ]:
if not df_corr.empty:
    heatmap_pairs = [
        ("dipoleAngle", "parallactic", "r:dipoleAngle vs η"),
        ("dipoleAngle", "H_hr", "r:dipoleAngle vs H (hr)"),
        ("dipoleAngle", "tan_z", "r:dipoleAngle vs tan z"),
        ("dipoleLength", "tan_z", "r:dipoleLength vs tan z"),
        ("dipoleLength", "airmass", "r:dipoleLength vs airmass"),
        ("dipoleLength", "H_hr", "r:dipoleLength vs |H| (hr)"),
        #        ("delta_signed", "parallactic", "Δ signed vs η"),
        #        ("delta_folded", "parallactic", "Δ folded vs η"),
    ]
    for y_lbl, x_lbl, title in heatmap_pairs:
        sub_heat = df_corr[(df_corr["x"] == x_lbl) & (df_corr["y"] == y_lbl)]
        if sub_heat.empty:
            continue
        pivot = sub_heat.pivot_table(index="field", columns="band", values="spearman_r").reindex(
            columns=BAND_ORDER
        )
        fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.65)))
        im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, label="Spearman ρ")
        ax.set_xticks(range(pivot.shape[1]))
        ax.set_xticklabels(pivot.columns.tolist())
        ax.set_yticks(range(pivot.shape[0]))
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Band")
        ax.set_ylabel("DDF")
        ax.set_title(f"Spearman ρ — {title}")
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
        plt.tight_layout()
        savefig(
            f"heatmap_{title.lower().replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}"
        )
        plt.show()

## 22. Summary

### Columns added in §4

| Column | Definition | Range |
|--------|------------|-------|
| `parallactic_angle_deg` | η = arctan2(sin H, tan φ cos δ − sin δ cos H) | −180 … +180° |
| `hour_angle_hr` | H = LST − RA, wrapped to (−12h, +12h] | ±12 h |
| `hour_angle_deg` | H × 15 | ±180° |
| `azimuth_deg` | CW from North | 0 … 360° |
| `zenith_angle_deg` | 90° − altitude | 0 … 90° |
| `sin_zenith` | sin(z) — DCR amplitude proxy | 0 … 1 |
| `airmass` | 1/cos(z) | ≥ 1 |
| `delta_dipole_para` | r:dipoleAngle − η, wrapped to (−180°, +180°] | ±180° |
| `delta_dipole_para_folded` | min(|Δ|, 180°−|Δ|), folded to [0°, 90°] | 0 … 90° |

### Physical interpretation

| Result | Interpretation |
|--------|----------------|
| Rose diagrams of `r:dipoleAngle` and η both peak at ±90° | Both mark the zenith direction — atmospheric DCR origin |
| `delta_dipole_para` concentrated at 0° **and** ±180° | Dipole axis tracks η; sign ambiguity from AP algorithm |
| `delta_dipole_para_folded` peaks near 0° | Near-perfect alignment of dipole axis with parallactic direction |
| `r:dipoleLength` grows with sin z | DCR amplitude ∝ tan z ≈ sin z — consistent with atmospheric dispersion |
| `r:dipoleAngle` shows S-curve vs H | Expected from η(H, δ, φ) dependence |
| η vs H shows clean S-curve (§18) | Geometry correctly computed, consistent with astroplan |

All figures are saved to `figs_DIPOLES_05b/`.
